In [12]:
from Booleanize_global import *

In [13]:
import pandas as pd
import os
import numpy as np
from Booleanize_global import *

csv_dir = "fractal_maps_isic"
base_dir = "/home/ubuntu/Downloads/DS_ISIC2019"

train_df = pd.read_csv(os.path.join(base_dir, "train_mapping.csv"))
test_df  = pd.read_csv(os.path.join(base_dir, "test_mapping.csv"))

In [14]:
def get_csv_name(split, label, fname):
    name = os.path.splitext(fname)[0]
    return f"{split}_{label}_{name}_fd.csv"

train_files = []
test_files = []

for _, row in train_df.iterrows():
    csv_name = get_csv_name("train", row["label"], row["filename"])
    train_files.append(csv_name)

for _, row in test_df.iterrows():
    csv_name = get_csv_name("test", row["label"], row["filename"])
    test_files.append(csv_name)

print("Train:", len(train_files))
print("Test:", len(test_files))


Train: 20264
Test: 5067


In [15]:
missing = []

for f in train_files + test_files:
    if not os.path.exists(os.path.join(csv_dir, f)):
        missing.append(f)

print("Missing files:", len(missing))

Missing files: 0


In [16]:
all_train_values = []

for f in train_files:
    path = os.path.join(csv_dir, f)

    vals = np.loadtxt(path, delimiter=",").flatten()
    all_train_values.extend(vals)

print("Collected training fractal values:", len(all_train_values))

Collected training fractal values: 65837736


In [17]:
discretizer = fit_global_bins(all_train_values, n_bins=5)

print("\nGlobal Quantile Bin Boundaries:")
bins = discretizer.bin_edges_[0]

for i in range(len(bins)-1):
    print(f"Bin {i}: {bins[i]:.5f} → {bins[i+1]:.5f}")


Global Quantile Bin Boundaries:
Bin 0: -0.16992 → 1.54057
Bin 1: 1.54057 → 1.93860
Bin 2: 1.93860 → 2.00000
Bin 3: 2.00000 → 2.43296
Bin 4: 2.43296 → 4.83289


/home/ubuntu/local/fractal_env/lib/python3.12/site-packages/sklearn/preprocessing/_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


In [18]:
csv_files = [f for f in os.listdir(csv_dir) if f.endswith(".csv")]
booleanized_data = {}

for idx, f in enumerate(csv_files):

    path = os.path.join(csv_dir, f)
    vals = np.loadtxt(path, delimiter=",").flatten()

    booleanized_data[f] = booleanize_array(vals, discretizer).flatten()

    if (idx + 1) % 2000 == 0:
        print(f"{idx+1}/{len(csv_files)} processed")

print("Booleanization completed.")

2000/25331 processed
4000/25331 processed
6000/25331 processed
8000/25331 processed
10000/25331 processed
12000/25331 processed
14000/25331 processed
16000/25331 processed
18000/25331 processed
20000/25331 processed
22000/25331 processed
24000/25331 processed
Booleanization completed.


In [19]:
X_train, Y_train = [], []
X_test, Y_test = [], []

# build label map (robust)
label_map = {}

for _, row in train_df.iterrows():
    key = get_csv_name("train", row["label"], row["filename"])
    label_map[key] = row["label"]

for _, row in test_df.iterrows():
    key = get_csv_name("test", row["label"], row["filename"])
    label_map[key] = row["label"]

# fill arrays
for f in train_files:
    if f in booleanized_data:
        X_train.append(booleanized_data[f])
        Y_train.append(label_map[f])

for f in test_files:
    if f in booleanized_data:
        X_test.append(booleanized_data[f])
        Y_test.append(label_map[f])

X_train = np.array(X_train)
X_test  = np.array(X_test)
Y_train = np.array(Y_train)
Y_test  = np.array(Y_test)

print("Shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

Shapes:
X_train: (20264, 16245)
X_test : (5067, 16245)


In [20]:
np.save('X_train.npy', X_train)
np.save('X_test.npy', X_test)
np.save('Y_train.npy', Y_train)
np.save('Y_test.npy', Y_test)

print("\n✅ Saved: X_train.npy, X_test.npy, Y_train.npy, Y_test.npy")


✅ Saved: X_train.npy, X_test.npy, Y_train.npy, Y_test.npy


In [21]:
X_train[5]

array([0, 0, 0, ..., 0, 0, 1], shape=(16245,))

In [22]:
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(20264, 16245) (20264,)
(5067, 16245) (5067,)


In [23]:
print("Train:", np.bincount(Y_train))
print("Test:", np.bincount(Y_test))

Train: [13468  6796]
Test: [3390 1677]
